In [65]:
import os
import subprocess
import time
from clearml import Task, OutputModel
from ultralytics import YOLO
from loguru import logger
import getpass
from loguru import logger

yolo8_default_params = {
    'model_variant': 'yolov8n.pt', # 모델 크기 (n, s, m, l, x)
    'epochs': 100,                 # 총 학습 에폭
    'imgsz': 640,                  # 이미지 크기
    'batch': 16,                   # 배치 사이즈
    'lr0': 0.01,                   # 초기 학습률
    'patience': 50,                # Early stopping 대기 횟수
    'device': 0                    # GPU 번호 (또는 'cpu')
}

# 환경 변수 선언 (Configuration)
work_environ = os.environ.get("WORK_ENVIRON", "company")
phase = os.environ.get("PHASE", "dev")
epochs_cnt = os.environ.get("EPOCHS_CNT", 1)
docker_image = os.environ.get("DOCKER_IMAGE", "172.16.11.236:5000/spire/python:3.12-bullseye")
task_version = os.environ.get("TASK_VERSION", "1.0")
data_duration = os.environ.get("DATA_DURATION", "24")
project_name = os.environ.get("PROJECT_NAME", "vision")
task_name = os.environ.get("TASK_NAME", "cat_dog")
bucket_name = os.environ.get("BUCKET_NAME", "clearml-data")
dataset_name = os.environ.get("DATASET_NAME", task_name)
classes = os.environ.get("CLASSES", "cat, dog").split(',')
params = os.environ.get("PARAMS", yolo8_default_params)

# PROJECT_NAME = "vision"
# DATASET_NAME = "cat_dog"
# TASK_NAME = "cat_dog"
# BUCKET_NAME = "clearml-data"
# CLASSES = ["cat", "dog"]
logger.info(f'work_environ  : {work_environ}')
logger.info(f'phase         : {phase}')
logger.info(f'epochs_cnt    : {epochs_cnt}')
logger.info(f'docker_image  : {docker_image}')
logger.info(f'task_version  : {task_version}')
logger.info(f'data_duration : {data_duration}')
logger.info(f'project_name  : {project_name}')
logger.info(f'task_name     : {task_name}')
logger.info(f'bucket_name   : {bucket_name}')
logger.info(f'dataset_name  : {dataset_name}')
logger.info(f'classes       : {classes}')
logger.info(f'params        : {params}')

if work_environ == "home":
    OBJECT_STORAGE_ENDPOINT = 'http://192.168.0.83:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = '7NFFU2I15W8ASBKI4J7T'
    AWS_ACCESS_SECRET_KEY = 'JGl1E8nCeItLJUvrn48y9FzdhZ+nU1+q95NsmxRH'
    AWS_REGION = 'ap-northeast-2'
else:
    OBJECT_STORAGE_ENDPOINT = 'http://172.16.11.235:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = 'QQY84JF5HCFNC814TE75'
    AWS_ACCESS_SECRET_KEY = 'TOzA6qIU0smDLZhQFS7N8jRUVCB+RbdYoyhtJ3Ma'
    AWS_REGION = 'ap-northeast-2'
    # Clearml 정보
    os.environ['CLEARML_WEB_HOST']='http://172.16.8.168:8080'
    os.environ['CLEARML_API_HOST']='http://172.16.8.168:8008'
    os.environ['CLEARML_FILES_HOST']='http://172.16.8.168:8081'
    os.environ['CLEARML_API_ACCESS_KEY']='NA4TVJLF4MNFPAT8JCSYBOVN0QASF2'
    os.environ['CLEARML_API_SECRET_KEY']='g3ur38Bn2GRwzTGDfcEGJy30iPv0Wx43TYDNhN4S4HjVQn5KX6OIpUbCFwTF2uOCQ3c'
    # YOLO의 자동 ClearML 로깅 비활성화
    os.environ['CLEARML_REGISTER_IGNORE'] = 'True'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_VCS_AUTO_CHECKOUT'] = '0'
    os.environ['CLEARML_VCS_IGNORE_EXTENSIONS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    # Git 정보를 찾지 않도록 설정
    os.environ['CLEARML_SKIP_GIT_CHECK'] = '1'
    # 원격지에서 실행 시 Git clone을 시도하지 않음
    os.environ['CLEARML_FORCE_STORE_DIFF'] = '1'


2026-05-15 10:41:30.721 | INFO     | __main__:<module>:39 - work_environ  : company
2026-05-15 10:41:30.724 | INFO     | __main__:<module>:40 - phase         : dev
2026-05-15 10:41:30.726 | INFO     | __main__:<module>:41 - epochs_cnt    : 1
2026-05-15 10:41:30.729 | INFO     | __main__:<module>:42 - docker_image  : 172.16.11.236:5000/spire/python:3.12-bullseye
2026-05-15 10:41:30.731 | INFO     | __main__:<module>:43 - task_version  : 1.0
2026-05-15 10:41:30.732 | INFO     | __main__:<module>:44 - data_duration : 24
2026-05-15 10:41:30.733 | INFO     | __main__:<module>:45 - project_name  : vision
2026-05-15 10:41:30.735 | INFO     | __main__:<module>:46 - task_name     : cat_dog
2026-05-15 10:41:30.736 | INFO     | __main__:<module>:47 - bucket_name   : clearml-data
2026-05-15 10:41:30.737 | INFO     | __main__:<module>:48 - dataset_name  : cat_dog
2026-05-15 10:41:30.739 | INFO     | __main__:<module>:49 - classes       : ['cat', ' dog']
2026-05-15 10:41:30.741 | INFO     | __main__

In [66]:
def mount_minio_vfs(bucket_name, project_name, dataset_name,
                    mount_path, aws_access_key_id, aws_access_secret_key,
                    aws_region, objectstorage_endpoint):
    # rclone 설정 (환경 변수 방식)
    os.environ["RCLONE_CONFIG_MYMINIO_TYPE"] = "s3"
    os.environ["RCLONE_CONFIG_MYMINIO_PROVIDER"] = "Minio"
    os.environ["RCLONE_CONFIG_MYMINIO_ACCESS_KEY_ID"] = aws_access_key_id
    os.environ["RCLONE_CONFIG_MYMINIO_SECRET_ACCESS_KEY"] = aws_access_secret_key
    os.environ["RCLONE_CONFIG_MYMINIO_ENDPOINT"] = objectstorage_endpoint

    os.makedirs(mount_path, exist_ok=True)

    # rclone 마운트 실행 (백그라운드)
    # --vfs-cache-mode minimal: 로컬에 복제하지 않고 실시간 읽기
    mount_cmd = [
        "rclone", "mount", f"minio_s3:{bucket_name}/{project_name}/{dataset_name}", mount_path,
        "--vfs-cache-mode", "full",
        "--allow-other",
        "--vfs-read-chunk-size", "1M",
        "--s3-region", aws_region,
        "--daemon"
    ]
    subprocess.run(mount_cmd, check=True)
    # 마운트 완료 대기
    for _ in range(10):
        if os.path.ismount(mount_path):
            logger.info(f"MinIO mounted at {mount_path}")
            return
        time.sleep(1)
    logger.error("Mounting failed")
    raise Exception("Mounting failed")

In [67]:
from pathlib import Path
import yaml
from pathlib import Path

class RcloneYoloDatasetManager:
    def __init__(self, mount_path, project_name, dataset_name, classes):
        """
        :param mount_path: rclone이 마운트된 로컬 경로 (예: '/mnt/minio_data/project1')
        :param project_name: project name
        :param dataset_name: dataset name
        """
        self._mount_path = Path(mount_path).absolute()
        self._project_name = project_name
        self._dataset_name = dataset_name
        self._classes = classes
        
        if not self._mount_path.exists():
            raise FileNotFoundError(f"마운트 경로를 찾을 수 없습니다: {self._mount_path}")
        self._file_list = []
        self._refresh_file_list()
        
    def _refresh_file_list(self):
        """마운트된 경로 내의 모든 파일 목록을 스캔합니다."""
        # rclone 마운트 특성상 rglob은 네트워크 오버헤드가 발생할 수 있으므로 주의 필요
        self._file_list = [
            f for f in self._mount_path.rglob('*') if f.is_file()
        ]
        logger.info(f"총 {len(self._file_list)}개의 파일을 로드했습니다.")

    def get_file_path(self, index):
        """인덱스로 파일의 전체 경로를 반환합니다."""
        return self._file_list[index]

    def read_data(self, index):
        """파일을 읽어 데이터를 반환합니다 (예: 텍스트 파일 기준)."""
        file_path = self.get_file_path(index)
        try:
            with open(file_path, 'rb') as f:
                return f.read()
        except Exception as e:
            logger.error(f"파일 읽기 오류 ({file_path}): {e}")
            return None

    def __len__(self):
        return len(self._file_list)

    def generate_yaml(self):
        """
        YOLOv8/v10/v11 등에서 사용하는 dataset.yaml 파일을 생성합니다.

        :param class_names: 클래스 이름 리스트 (예: ['cat', 'dog'])
        :param train_dir: 마운트 경로 내 train 폴더명
        :param val_dir: 마운트 경로 내 val 폴더명
        """
        data_config = {
            'path': str(self._mount_path),  # 데이터셋 루트 경로
            'train': str(self._mount_path), # 학습 이미지 경로
            'val': str(self._mount_path),   # 검증 이미지 경로
            'names': {i: name for i, name in enumerate(self._classes)}
        }
        
        yaml_path = self._mount_path / f"{self._dataset_name}.yaml"
        
        try:
            with open(yaml_path, 'w', encoding='utf-8') as f:
                yaml.dump(data_config, f, default_flow_style=False, sort_keys=False)
            logger.info(f"YAML 파일 생성 완료: {yaml_path}")
            return yaml_path
        except Exception as e:
            logger.error(f"YAML 생성 실패: {e}")
            return None


In [70]:
# 이전 모델 정보 다운 로드
def get_pretrained_weigth(project_name, task_name):
    """
    마지막으로 훈련에 성공한 모델의 가중치 파일 다운 받습니다.
    """
    tasks = Task.query_tasks(
        project_name=project_name,
        task_name=task_name,
        task_filter={
            'status': ['completed'],
            'order_by': ['-created']  # Ensures the most recently updated is first
        }
    )
    logger.info(f'tasks\'len = {len(tasks)}')
    model_weight_local_path = None
    if len(tasks) > 0:
        last_task_id= tasks[0]
        logger.info(f'last_task_id = {last_task_id}')

        last_task = Task.get_task(task_id=last_task_id)
        models = last_task.get_models()

        # 3. Access a specific model (e.g., the first output model)
        if models.get('output'):
            latest_model = models['output'][0]
            # Download the weights to a local path
            model_weight_local_path = latest_model.get_local_copy()
    logger.info(f"Model downloaded to #3: {model_weight_local_path}")
    return model_weight_local_path

2026-05-15 10:43:03.993 | INFO     | __main__:<module>:12 - last_task_id = dbfbdf57daf043ceb6de72b5c947fd63
2026-05-15 10:43:04.099 | INFO     | __main__:<module>:23 - Model downloaded to: /home/jupyter/jupyter-workspace/vision/runs/detect/train11/weights/best.pt


In [ ]:
task = Task.get_task(project_name="Ultralytics", task_name="train")
if task:
    task.close()

task = Task.init(project_name=project_name,
                 task_name=task_name,
                 reuse_last_task_id=False,
                 continue_last_task=False,
                 task_type=Task.TaskTypes.training)
if len(params.keys()) > 0:
    task.connect(params)
task.set_parameter('version', task_version)
task.set_task_type('training')
ia_task_initiated = False

requirements = ['ultralytics', 'boto3',
                'matplotlib', 'seaborn',
                'albumentations', 'tqdm']
task.set_packages(requirements)

logger.info(f'run in phase = {phase}')
if phase == "prod":
    task.set_base_docker(
        docker_image=docker_image,  # 원격에서 실행할 베이스 이미지
        docker_arguments="--privileged --device /dev/fuse"  # 도커 실행 인자
    )

    task.execute_remotely(queue_name='services',
                          clone=True,
                          exit_process=False)

    from clearml.config import running_remotely

    if not running_remotely():
        # 로컬 Parent 프로세스 영역
        logger.info("Parent: 자식 태스크를 원격 에이전트에 전달했습니다. [current: {task.id}]")
        task.close()
        task.delete(delete_artifacts_and_models=True)
        exit()

mount_path = f"minio_mnt/{project_name}/{dataset_name}"
os.makedirs(mount_path, exist_ok=True)
mount_minio_vfs(bucket_name=bucket_name, project_name=project_name,
                dataset_name=dataset_name, mount_path=mount_path,
                aws_access_key_id=AWS_ACCESS_KEY_ID,
                aws_access_secret_key=AWS_ACCESS_SECRET_KEY,
                aws_region=AWS_REGION,
                objectstorage_endpoint=OBJECT_STORAGE_ENDPOINT)
dir_list = os.listdir(mount_path)
# ClearML 학습 코드 시작 부분에 추가
classes = ["cat", "dog"] # 실제 클래스 순서대로 입력
logger.info(f'mount_path = {mount_path}')
manager = RcloneYoloDatasetManager(mount_path=mount_path,
                                   project_name="vision",
                                   dataset_name="cat_dog",
                                   classes=classes
                                  )
yaml_file = manager.generate_yaml()
logger.info(f'yaml_file = {yaml_file}')

model_weigth_local_path = get_pretrained_weigth(project_name=project_name, task_name=task_name)
    
if len(manager) > 0:
    first_data = manager.read_data(0)
    logger.info(f"첫 번째 파일 크기: {len(first_data)} bytes")

if model_weigth_local_path is None:
    model = YOLO('yolov8n.pt')
else:
    model = YOLO(model_weigth_local_path)

model.train(data=str(yaml_file), epochs=epochs_cnt)
logger.info(f'completed to train. [model={model}]')

In [76]:
# 2. ONNX 형식으로 내보내기 (Export)
# format='onnx'로 설정하면 best.onnx 파일이 생성됩니다.
onnx_path = model.export(format='onnx', dynamic=True) 
output_model = OutputModel(task=task)

# 로컬에 있는 파일을 서버로 업로드
output_model.update_weights(
    weights_filename=onnx_path # 업로드할 파일 경로
)
# output_model.set_metadata('threshold', str(best_threshold))
output_model.set_metadata('version', '1.0')
output_model.set_metadata('format', 'onnx')
logger.info(f"모델이 저장되었습니다. [{onnx_path}]")

# 4. ClearML Artifact에 등록 (선택 사항)
task.upload_artifact(
    name='onnx_model',
    artifact_object=onnx_path
)

Ultralytics 8.4.33 🚀 Python-3.10.20 torch-2.11.0+cu130 CPU (Intel Xeon Silver 4208 CPU @ 2.10GHz)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/home/jupyter/jupyter-workspace/vision/runs/detect/train12/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.1 MB)

ONNX: starting export with onnx 1.21.0 opset 20...
ONNX: slimming with onnxslim 0.1.90...


/opt/jupyter/kernel/python3_10/lib/python3.10/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset11.py:954: UserWarning:

Exporting aten::index operator of advanced indexing in opset 20 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.



ONNX: export success ✅ 3.4s, saved as '/home/jupyter/jupyter-workspace/vision/runs/detect/train12/weights/best.onnx' (9.6 MB)

Export complete (3.7s)
Results saved to /home/jupyter/jupyter-workspace/vision/runs/detect/train12/weights
Predict:         yolo predict task=detect model=/home/jupyter/jupyter-workspace/vision/runs/detect/train12/weights/best.onnx imgsz=640 
Validate:        yolo val task=detect model=/home/jupyter/jupyter-workspace/vision/runs/detect/train12/weights/best.onnx imgsz=640 data=/home/jupyter/jupyter-workspace/vision/minio_mnt/vision/cat_dog/cat_dog.yaml  
Visualize:       https://netron.app


2026-05-15 10:54:41.731 | INFO     | __main__:<module>:13 - 모델이 저장되었습니다. [/home/jupyter/jupyter-workspace/vision/runs/detect/train12/weights/best.onnx]


True

In [77]:
##########################
## finalization
##########################

logger.info(f'umount path. [{mount_path}].')
subprocess.run(["umount", mount_path], check=True)

import shutil
import time
from pathlib import Path
logger.info(f'wait to complete unmount for 2 seconds.')
time.sleep(2)
path = Path('minio_mnt')
if path.exists() and path.is_dir():
    shutil.rmtree(path)
logger.info(f'The mounted folder is removed.')
task.close()
logger.info("completed to train...")

2026-05-15 10:54:43.988 | INFO     | __main__:<module>:5 - umount path. [minio_mnt/vision/cat_dog].
2026-05-15 10:54:44.013 | INFO     | __main__:<module>:11 - wait to complete unmount for 2 seconds.
2026-05-15 10:54:46.018 | INFO     | __main__:<module>:16 - The mounted folder is removed.
2026-05-15 10:54:52.356 | INFO     | __main__:<module>:18 - completed to train...
